# 問題
単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [1]:
# 単語埋め込み語彙の作成
import numpy as np
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
vocab = list(model.key_to_index.keys())
d_emb = model.vector_size
V = len(vocab) + 1

# 埋め込み行列の初期化
E = np.zeros((V, d_emb), dtype=np.float32)

# インデックス対応表
word2id = {'<PAD>': 0}
id2word = {0: '<PAD>'}

# 行列にベクトルを格納
for i, word in enumerate(vocab, start=1):
    E[i] = model[word]
    word2id[word] = i
    id2word[i] = word

In [22]:
import torch

def sst_build_answer_list(path: str):
    """
    SST-2のTSVを読み込み、
      - 文章→単語分割
      - word2idでID列に変換（辞書にない語は除外）
      - パディングは行わず、可変長のTensorリストとして保持
    事前条件:
      - グローバルに `word2id` が存在
    返り値:
      - answer_list: List[Dict[str, Any]]
          text: 元文（str）
          label: torch.long テンソル（スカラ）
          input_ids: torch.long テンソル（長さ=文ごとに可変）
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            row = raw.strip().split("\t")
            if len(row) < 2:
                continue
            label = row[1]
            if label not in ("0", "1"):
                continue
            text = row[0]
            tokens = text.strip().split()
            examples.append({"text": text, "tokens": tokens, "label": int(label)})

    # ID化（辞書外は除外）
    kept_examples = []
    for ex in examples:
        ids = [word2id[w] for w in ex["tokens"] if w in word2id]
        if ids:
            ex["input_ids"] = torch.tensor(ids, dtype=torch.long)
            kept_examples.append(ex)
        else:
            print(f'"{ex["text"]}" は辞書に該当単語がなくスキップしました')
            pass

    # answer_list生成
    answer_list = []
    for ex in kept_examples:
        answer_list.append({
            "text": ex["text"],
            "label": torch.tensor(ex["label"], dtype=torch.long),
            "input_ids": ex["input_ids"]
        })

    return answer_list

In [23]:
# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"
dev_71 = sst_build_answer_list(path_dev)
train_71 = sst_build_answer_list(path_train)

"oh-so-important " は辞書に該当単語がなくスキップしました
"beloved-major " は辞書に該当単語がなくスキップしました
"light-hearted " は辞書に該当単語がなくスキップしました
"time-consuming " は辞書に該当単語がなくスキップしました
"fresh-faced " は辞書に該当単語がなくスキップしました
"sleep-inducing " は辞書に該当単語がなくスキップしました
"self-absorbed " は辞書に該当単語がなくスキップしました
"good-natured " は辞書に該当単語がなくスキップしました
"queasy-stomached " は辞書に該当単語がなくスキップしました
"none-too-original " は辞書に該当単語がなくスキップしました
"well-intentioned " は辞書に該当単語がなくスキップしました
"big-hearted and " は辞書に該当単語がなくスキップしました
"well-meant " は辞書に該当単語がなくスキップしました
"kid-empowerment " は辞書に該当単語がなくスキップしました
"a rip-off " は辞書に該当単語がなくスキップしました
"therapy-dependent flakeball " は辞書に該当単語がなくスキップしました
"good-looking " は辞書に該当単語がなくスキップしました
"soon-to-be-forgettable " は辞書に該当単語がなくスキップしました
", cliche-ridden " は辞書に該当単語がなくスキップしました
"big-hearted " は辞書に該当単語がなくスキップしました
"'' has-been " は辞書に該当単語がなくスキップしました
"bad-movie " は辞書に該当単語がなくスキップしました
"linklater " は辞書に該当単語がなくスキップしました
"a re-hash " は辞書に該当単語がなくスキップしました
"well-put-together " は辞書に該当単語がなくスキップしました
"mind-numbing " は辞書に該当単語がなくスキップしました
"cheap-looking " は辞

In [34]:
import torch
from torch import nn
import numpy as np

def build_avg_features(examples, E, pad_id=0):
    # numpy配列ならtorch.Tensorに変換
    if isinstance(E, np.ndarray):
        E = torch.tensor(E, dtype=torch.float32)

    X, y = [], []
    for ex in examples:
        ids = ex["input_ids"]
        if pad_id is not None:
            ids = ids[ids != pad_id]
        if ids.numel() == 0:
            v = torch.zeros(E.size(1))
        else:
            v = E[ids].mean(dim=0)  # (len(ids), d) → (d,)
        X.append(v)
        y.append(ex["label"].float())

    return torch.stack(X, dim=0), torch.stack(y, dim=0)

X_train, y_train = build_avg_features(train_71, E)
X_dev, y_dev = build_avg_features(dev_71,   E)



In [ ]:
model = nn.Linear(X_train.size(1), 1)     # w,b を内包
crit  = nn.BCEWithLogitsLoss()
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

for _ in range(20):
    opt.zero_grad()
    logit = model(X_train).squeeze(1)     # (N,)
    loss  = crit(logit, y_train)
    loss.backward()
    opt.step()

with torch.no_grad():
    prob = torch.sigmoid(model(X_dev).squeeze(1))
    pred = (prob >= 0.5).long()
    acc  = (pred == y_dev.long()).float().mean().item()
print(f"dev acc: {acc:.3f}")

# 重みベクトルとバイアス
w = model.weight.detach().squeeze(0)  # (d,)
b = model.bias.detach().item()

dev acc: 0.517
